### Lab 5: Processing Incremental Updates with PySpark Structured Streaming and Delta tables
In this lab you'll apply your knowledge of PySpark and structured streaming to implement a simple multi-hop (medallion) architecture.

#### 1.0. Import Required Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
os.environ["JAVA_HOME"] = "/Users/tianyao/Library/Java/JavaVirtualMachines/ms-21.0.10/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))

import pyspark
print("PySpark version =", pyspark.__version__)

import delta

import sys
import json
import shutil
import time

import pyspark
from delta import *
from pyspark.sql.functions import *
from pyspark.sql.types import *


/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark
JAVA_HOME = /Users/tianyao/Library/Java/JavaVirtualMachines/ms-21.0.10/Contents/Home
PySpark version = 3.5.8


#### 2.0. Instantiate Global Variables

In [2]:
# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'lab_data')
data_dir = os.path.join(base_dir, 'retail-org')
customers_stream_dir = os.path.join(data_dir, 'customers')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "customers_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
database_dir = os.path.join(sql_warehouse_dir, dest_database)

customers_output_bronze = os.path.join(database_dir, 'customers_bronze')
customers_output_silver = os.path.join(database_dir, 'customers_silver')
customers_output_gold = os.path.join(database_dir, 'customers_gold')

#### 3.0. Define Global Functions

In [3]:
def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"

#### 4.0. Create a New Spark Session

In [4]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"
shuffle_partitions = int(os.cpu_count())

builder = pyspark.sql.SparkSession.builder \
    .appName('PySpark Customers Delta Table in Juptyer')\
    .master(worker_threads)\
    .config('spark.driver.memory', '4g') \
    .config('spark.executor.memory', '2g')\
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .config('spark.sql.debug.maxToStringFields', 50) \
    .config('spark.sql.shuffle.partitions', shuffle_partitions) \
    .config('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .config('spark.sql.streaming.schemaInference', 'true') \
    .config('spark.sql.warehouse.dir', database_dir) \
    .config('spark.streaming.stopGracefullyOnShutdown', 'true')

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

26/04/05 14:37:38 WARN Utils: Your hostname, Tianyaos-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.41.1.125 instead (on interface en0)
26/04/05 14:37:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/tianyao/.ivy2/cache
The jars for the packages stored in: /Users/tianyao/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d7438091-04e8-4f97-9162-896ee66cad42;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 64ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	------------------------------------------------------------------

:: loading settings :: url = jar:file:/opt/anaconda3/envs/pysparkenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/05 14:37:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### 5.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory Structure to Ensure Idempotency

In [5]:
remove_directory_tree(database_dir)

"Directory '/Users/tianyao/Desktop/DS-2002-main copy/04-PySpark/spark-warehouse/customers_dlh' has been removed successfully."

#### 6.0. Bronze Table: Ingest and Stage Data
This lab uses a collection of customer-related CSV data found in *`../04-PySpark/lab_data/retail-org/customers/`*. 
<br>This is available to you by way of the `customers_stream_dir` variable.

##### 6.1. Read this data into a Stream using schema inference
- Use a **`_checkpoint`** folder and the **`schemaLocation`** option to store the schema info in a dedicated folder for **`customers`**.
- Set the **`maxFilesPerTrigger`** option to **`1`**.
- Set the **`inferSchema`** and **`header`** options to **`true`**.
- Use **`.csv()`** to specify the source directory.

In [6]:
customers_checkpoint_bronze = os.path.join(customers_output_bronze, '_checkpoint')

df_customers_bronze = (
    spark.readStream \
    # TODO: Configurations
    .option("schemaLocation", customers_checkpoint_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv(customers_stream_dir)
)

df_customers_bronze.isStreaming

True

In [7]:
df_customers_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- tax_id: double (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- valid_from: integer (nullable = true)
 |-- valid_to: double (nullable = true)
 |-- units_purchased: double (nullable = true)
 |-- loyalty_segment: integer (nullable = true)



26/04/05 14:37:49 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


##### 6.2. Stream the raw data to a Delta table.
 - Use the **`delta`** format.
 - Use the **`append`** output mode.
 - Use **`customers_bronze`** as the **`queryName`**.
 - Use **`availableNow = True`** for the **`trigger`**
 - Use the **`_checkpoint`** folder with the **`checkpointLocation`** option.

In [8]:
customers_bronze_query = (
    df_customers_bronze \
    .writeStream \
    # TODO: Configurations
    .format("delta") \
    .outputMode("append") \
    .queryName("customers_bronze") \
    .trigger(availableNow = True) \
    .option("checkpointLocation", customers_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(customers_output_bronze)
)

In [9]:
customers_bronze_query.awaitTermination()

##### 6.3. Create a Streaming Temporary View named **`customers_bronze_temp`**
- Use the **`delta`** format.
- Set the **`inferSchema`** option to **`true`**
- Load the data from the output of the **`bronze`** delta table (**`customers_output_bronze`**)

In [10]:
(spark.readStream \
    # TODO: Configurations
     .format("delta") \
     .option("inferSchema", "true") \
     .load(customers_output_bronze) \
     .createOrReplaceTempView("customers_bronze_temp")
)

##### 6.4. Clean and Enhance the Data
Use the CTAS syntax to define a new streaming view called **`bronze_enhanced_temp`** that does the following:
* Omits records with a null **`postcode`** (set to zero)
* Inserts a column called **`receipt_time`** containing a current timestamp using the **`current_timestamp()`** function.
* Inserts a column called **`source_file`** containing the input filename using the **`imput_file_name()`** function.

In [13]:
sql_bronze_temp = """
   CREATE OR REPLACE TEMPORARY VIEW bronze_enhanced_temp AS
       SELECT *
           , current_timestamp() receipt_time
           , input_file_name() source_file
        FROM customers_bronze_temp
        WHERE postcode > 0
"""
spark.sql(sql_bronze_temp)

DataFrame[]

#### 7.0. Silver Table
##### 7.1. Stream the data from **`bronze_enhanced_temp`** to a **`Delta`** table named **`customers_silver`**.
 - Use the **`append`** output mode.
 - Use **`customers_silver`** as the **`queryName`**.
 - Use **`availableNow = True`** for the **`trigger`**
 - Use a **`_checkpoint`** folder with the **`checkpointLocation`** option to store the schema info in a dedicated folder for **`customers`**.

In [14]:
customers_checkpoint_silver = os.path.join(customers_output_silver, '_checkpoint')

customers_silver_query = (
    spark.table("bronze_enhanced_temp")
        .writeStream
        .format("delta")
        .outputMode("append")
        .queryName("customers_silver")
        .trigger(availableNow=True)
        .option("checkpointLocation", customers_checkpoint_silver)
        .start(customers_output_silver)
)

In [15]:
customers_silver_query.awaitTermination()

##### 7.2. Create a Streaming Temporary View
- Create another streaming temporary view named **`customers_silver_temp`** for the **`customers_silver`** table so we can perform business-level queries using SQL.

In [16]:
spark.readStream \
    .format("delta") \
    .load(customers_output_silver) \
    .createOrReplaceTempView("customers_silver_temp")

#### 8.0. Gold Table
##### 8.1. Use the CTAS syntax to define a new streaming view called **`customer_count_by_state_temp`** that does the following:
- Reads data from the **`customers_silver_temp`** temporary view created in the preceding step.
- Selects the **`state`** along with the number of customers per (grouped by) state.

In [17]:
sql_gold_temp = """
CREATE OR REPLACE TEMPORARY VIEW customer_count_by_state_temp AS
SELECT
    state,
    COUNT(*) AS customer_count
FROM customers_silver_temp
GROUP BY state
"""
spark.sql(sql_gold_temp)

DataFrame[]

##### 8.2. Stream the data from the **`customer_count_by_state_temp`** view to a Delta table called **`customer_count_by_state_gold`**.
- Use the **`complete`** output mode because aggregations like **`count()`** and sorting cannot operate on *unbounded* datasets.  
- Use a **`_checkpoint`** folder with the **`checkpointLocation`** option and a dedicated folder for **`customers`** as the checkpoint path.

In [21]:
customer_count_by_state_checkpoint = os.path.join(customers_output_gold, '_checkpoint')
customer_count_by_state_output = customers_output_gold

customer_count_by_state_gold_query = (
    spark.table("customer_count_by_state_temp")
        .writeStream
        .format("delta")
        .outputMode("complete")
        .queryName("customer_count_by_state_gold")
        .trigger(availableNow=True)
        .option("checkpointLocation", customer_count_by_state_checkpoint)
        .start(customer_count_by_state_output)
)

In [22]:
customer_count_by_state_gold_query.awaitTermination()

#### 9.0. Query the Results
- Query the **`customer_count_by_state_gold`** table (this will not be a streaming query).
- Select the **`state`** and **`customer_count`** columns.
- Sort the results by **`customer_count`** in descending order (i.e., from highest to lowest).

In [23]:
sql_customer_count_query = """
SELECT
    state,
    customer_count
FROM delta.`{}` 
ORDER BY customer_count DESC
""".format(customer_count_by_state_output)

spark.sql(sql_customer_count_query).toPandas()

,state,customer_count
0,NY,3363
1,CA,2874
2,FL,2517
3,OH,1914
4,MA,1889
5,NJ,1503
6,IN,1105
7,MI,1079
8,NC,1011
9,WI,938


In [24]:
spark.stop()

26/04/05 15:01:10 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:642)
	at java.base/java.lang.Thread.